# Config

In [1]:
# Cell 1 — Parameters. Change experimental values here only.
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset')
WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'FOA-Attack'
OUTPUT_DIR = WORK_ROOT / 'foa_attack_output'
HF_HOME = WORK_ROOT / 'hf_cache'

# Official implementation used for parity. Its resolved revision is recorded automatically.
FOA_REPO_URL = 'https://github.com/jiaxiaojunQAQ/FOA-Attack.git'

In [2]:
# Paper
PAPER_NUM_SAMPLES = 100
PAPER_STEPS = 300
PAPER_EPSILON = 16.0       # pixel space [0, 255]
PAPER_ALPHA = 1.0          # pixel space [0, 255]
PAPER_INPUT_RES = 224
PAPER_CROP_SCALE = (0.5, 0.9)
PAPER_BACKBONES = ('B16', 'B32', 'Laion')
PAPER_CLUSTER_NUMBER = 10
PAPER_SEED = 2023

# Model 
NUM_SAMPLES = 10
STEPS = 300

# 'none' is parity mode. 'saliency' is an experimental XAI hook, not FOA.
# XAI_MODE = 'none'
# XAI_WEIGHT = 0.0
DEVICE = 'cuda:0'

assert DATASET_ROOT.exists(), 'Add Kaggle dataset: zintom/blackbox-adversarial-attack-dataset'
# assert XAI_MODE in {'none', 'saliency'}
print(f'Dataset: {DATASET_ROOT}\nOutput: {OUTPUT_DIR}\nDevice: {DEVICE}')

Dataset: /kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset
Output: /kaggle/working/foa_attack_output
Device: cuda:0


# Install package

In [3]:
# Cell 2 — Kaggle dependency command. Re-run after changing the Kaggle image.
import sys, subprocess
packages = [
    'transformers==4.49.0', 'tokenizers==0.21.0', 'huggingface-hub==0.28.1',
    'hydra-core==1.3.2', 'omegaconf==2.3.0', 'pytorch-lightning==2.5.0.post0',
    'wandb==0.19.1', 'ftfy==6.3.1',
    # 'kmeans-pytorch==0.3',     
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *packages], check=True)
print('Dependencies installed. Restart the Kaggle session if pip reports a Torch/Torchvision conflict.')

Dependencies installed. Restart the Kaggle session if pip reports a Torch/Torchvision conflict.


In [4]:
!pip install "git+https://github.com/subhadarship/kmeans_pytorch.git"

  Cloning https://github.com/subhadarship/kmeans_pytorch.git to /tmp/pip-req-build-cdfxnrjf
  Running command git clone --filter=blob:none --quiet https://github.com/subhadarship/kmeans_pytorch.git /tmp/pip-req-build-cdfxnrjf
  Resolved https://github.com/subhadarship/kmeans_pytorch.git to commit f7f36bd1cb4e3a761d73d584866d0a9c6b4d2805
  Preparing metadata (setup.py) ... done


# Clone Github

In [5]:
# Cell 3 — Fetch the original implementation and record source provenance.
import os, shutil, subprocess, sys
os.environ.update({'HF_HOME': str(HF_HOME), 'WANDB_MODE': 'disabled', 'TOKENIZERS_PARALLELISM': 'false'})
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', FOA_REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
FOA_SOURCE_REVISION = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('Official FOA source revision:', FOA_SOURCE_REVISION)

Cloning into '/kaggle/working/FOA-Attack'...


Official FOA source revision: 6aa9ad463a1e812b7f77ff2e5c4b1712e7c78c4b


# Library

In [6]:
# Cell 4 — Imports and reproducibility.
import json, random, hashlib, importlib.util
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import numpy as np
from PIL import Image
import torch
from torch import nn, Tensor
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from torchvision.utils import save_image
from tqdm.auto import tqdm
from kmeans_pytorch import kmeans

from surrogates import (
    ClipB16FeatureExtractor, ClipB32FeatureExtractor, ClipLaionFeatureExtractor,
    EnsembleFeatureExtractor_ot,
)

def set_foa_seed(seed: int = PAPER_SEED) -> None:
    """Set the same random/CUDA determinism controls as the original FOA script."""
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

set_foa_seed()
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print('Torch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
# print(inspect.signature(kmeans))
# inspect.signature(kmeans)

Torch: 2.11.0+cu128 | GPU: Tesla T4


# Data loader

In [7]:
# Cell 5 — Dataset discovery and preprocessing (editable research boundary).
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_images(path: Path) -> int:
    """Return recursive number of supported image files below ``path``."""
    return sum(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in path.rglob('*'))

def find_imagefolder(root: Path, expected_name: str) -> Path:
    """Find the largest ImageFolder named ``expected_name`` in a Kaggle dataset."""
    candidates = sorted((p for p in root.rglob(expected_name) if p.is_dir()), key=lambda p: -count_images(p))
    if not candidates: raise FileNotFoundError(f'Cannot find {expected_name} below {root}')
    return candidates[0]

def pil_to_foa_tensor(image: Image.Image) -> Tensor:
    """Convert PIL RGB image to the original FOA tensor in pixel range [0, 255]."""
    array = np.array(image, np.uint8, copy=True)
    return torch.from_numpy(array).permute(2, 0, 1).contiguous().to(torch.get_default_dtype())


# Preprocessing
def preprocess_image(input_res: int = PAPER_INPUT_RES) -> transforms.Compose:
    """Build the exact Resize → CenterCrop → RGB → pixel-tensor preprocessing of FOA."""
    return transforms.Compose([
        transforms.Resize(input_res, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(input_res), transforms.Lambda(lambda x: x.convert('RGB')),
        transforms.Lambda(pil_to_foa_tensor),
    ])

class ImageFolderWithPaths(datasets.ImageFolder):
    """ImageFolder that additionally returns the source image path for manifest export."""
    def __getitem__(self, index: int):
        """Return transformed image, class label, and original filesystem path."""
        image, label = super().__getitem__(index)
        return image, label, self.samples[index][0]

SOURCE_DIR, TARGET_DIR = find_imagefolder(DATASET_ROOT, 'bigscale'), find_imagefolder(DATASET_ROOT, 'target_images')
print('SOURCE_DIR:', SOURCE_DIR, count_images(SOURCE_DIR))
print('TARGET_DIR:', TARGET_DIR, count_images(TARGET_DIR))

SOURCE_DIR: /kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/bigscale 1000
TARGET_DIR: /kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images 1000


# Visualize dataset

In [8]:
def show_images(folder, n=12):
    paths = sorted([p for p in Path(folder).rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])[:n]
    fig, axes = plt.subplots(math.ceil(n / 4), 4, figsize=(16, math.ceil(n / 4) * 4))
    for ax, path in zip(axes.flatten(), paths):
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(path.name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [9]:
source_path = Path(str(SOURCE_DIR) + "/nips17")
target_path = Path(str(TARGET_DIR) + "/1")
source_image_path = [p for p in source_path.rglob('*')]
target_image_path = [p for p in target_path.rglob('*')]
# sorted(source_image_path, str(source_image_path))
# target_image_path
image_paths = sorted(source_image_path)

In [10]:
import math
import matplotlib.pyplot as plt

def show_images(source_image_path, n=12, cols=4, random_sample=False, seed=42, figsize_per_image=(4, 4)):
    """
    Display images recursively from a folder.

    Args:
        folder: Path to image folder.
        n: Number of images to display.
        cols: Number of columns.
        random_sample: Randomly choose images if True.
        seed: Random seed.
        figsize_per_image: Size of each displayed image.
    """
    if len(source_image_path) == 0:
        return

    # image_paths = sorted(source_image_path)
    image_paths = source_image_path

    if random_sample:
        random.seed(seed)
        image_paths = random.sample(image_paths, min(n, len(image_paths)))
    else:
        image_paths = image_paths[:n]

    rows = math.ceil(len(image_paths) / cols)

    fig, axes = plt.subplots(rows,cols,
        figsize=(
            cols * figsize_per_image[0],
            rows * figsize_per_image[1]
        )
    )

    # Make axes always iterable
    if rows == 1 and cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for ax, image_path in zip(axes, image_paths):
        image = Image.open(image_path).convert("RGB")

        ax.imshow(image)
        ax.set_title(image_path.name, fontsize=12)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"Displayed: {len(image_paths)} images")

In [11]:
# show_images(source_image_path)

In [12]:
# show_images(target_image_path)

In [13]:
# Cell 6 — FOA surrogate ensemble (unchanged original model choices).
BACKBONE_MAP = {'B16': ClipB16FeatureExtractor, 'B32': ClipB32FeatureExtractor, 'Laion': ClipLaionFeatureExtractor}

def build_foa_ensemble(backbones: Tuple[str, ...] = PAPER_BACKBONES, cluster_number: int = PAPER_CLUSTER_NUMBER):
    """Load original FOA surrogate extractors and its optimal-transport ensemble wrapper."""
    models = [BACKBONE_MAP[name]().eval().to(DEVICE).requires_grad_(False) for name in backbones]
    return EnsembleFeatureExtractor_ot(models, cluster_number=cluster_number), models

# Run only once; it downloads B16, B32 and LAION weights on the first Kaggle run.
ensemble_extractor, surrogate_models = build_foa_ensemble()
print('Loaded:', PAPER_BACKBONES)

Loaded: ('B16', 'B32', 'Laion')


# Attacking

## Loss function

In [14]:
# Cell 7 — FOA loss. This is the paper loss copied into an editable cell.
class FOAOptimalTransportLoss(nn.Module):
    """Original FOA global/local OT loss with its dynamic ensemble weighting (K=10 by default)."""
    def __init__(self, extractors: List[nn.Module], cluster_number: int = PAPER_CLUSTER_NUMBER):
        """Initialize FOA state for the frozen surrogate extractors and local cluster count."""
        super().__init__(); self.extractors = nn.ModuleList(extractors); self.cluster_number = cluster_number
        self.ground_truth, self.ground_truth_local, self.previous_loss_list = [], [], []

    @torch.no_grad()
    def set_ground_truth(self, x: Tensor) -> None:
        """Extract target global features and K-means local centers as FOA ground truth."""
        self.ground_truth.clear(); self.ground_truth_local.clear()
        for model in self.extractors:
            global_feature, local_feature = model.global_local_features(x.to(x.device))
            centers = self.cluster_centers(local_feature.squeeze(0), x.device).unsqueeze(0)
            self.ground_truth.append(global_feature); self.ground_truth_local.append(centers)

    def cluster_centers(self, embeddings: Tensor, device: torch.device) -> Tensor:
        """Compute original Euclidean K-means local centers (100 iteration limit)."""
        _, centers = kmeans(X=embeddings, num_clusters=self.cluster_number, distance='euclidean', device=device, iter_limit=100)
        # _, centers = kmeans(X=embeddings, num_clusters=self.cluster_number, distance='euclidean', device=device)
        return centers.to(device)

    def optimal_transport(self, source: Tensor, target: Tensor) -> Tensor:
        """Return FOA cosine-similarity optimal-transport score using original Sinkhorn settings."""
        similarity = torch.einsum('md,nd->mn', F.normalize(source, dim=1), F.normalize(target, dim=1)).contiguous()
        kernel = torch.exp(-(1 - similarity) / 0.1)
        u = torch.full((source.shape[0],), 1 / source.shape[0], dtype=similarity.dtype, device=similarity.device)
        v = torch.full((target.shape[0],), 1 / target.shape[0], dtype=similarity.dtype, device=similarity.device)
        with torch.no_grad(): transport = self.sinkhorn(kernel, u, v)
        if torch.isnan(transport).any(): raise FloatingPointError('FOA Sinkhorn returned NaN')
        return torch.sum(transport * similarity)

    def sinkhorn(self, kernel: Tensor, u: Tensor, v: Tensor) -> Tensor:
        """Run the original 100-step, 1e-2-threshold Sinkhorn normalization."""
        r, c = torch.ones_like(u), torch.ones_like(v)
        for _ in range(100):
            old_r = r; r = u / (kernel @ c.unsqueeze(-1)).squeeze(-1); c = v / (kernel.T @ r.unsqueeze(-1)).squeeze(-1)
            if (r - old_r).abs().mean().item() < 1e-2: break
        return torch.outer(r, c) * kernel

    def forward(self, global_features: Dict[int, Tensor], local_features: Dict[int, Tensor]) -> Tensor:
        """Compute original FOA global OT + 0.2 local OT and dynamic model weights."""
        losses = [self.optimal_transport(self.ground_truth[i], global_features[i].unsqueeze(0)) + 0.2 * self.optimal_transport(self.ground_truth_local[i].squeeze(0), local_features[i].squeeze(0)) for i in range(len(self.extractors))]
        if not self.previous_loss_list: self.previous_loss_list = [loss.detach() for loss in losses]
        weights = np.array([loss.item() / (prev.item() + 1e-8) for loss, prev in zip(losses, self.previous_loss_list)])
        weights = np.exp(weights) / np.exp(weights).sum() * len(weights)
        self.previous_loss_list = [loss.detach() for loss in losses]
        return sum(float(weights[i]) * loss for i, loss in enumerate(losses))

foa_loss = FOAOptimalTransportLoss(surrogate_models)
foa_loss

FOAOptimalTransportLoss(
  (extractors): ModuleList(
    (0): ClipB16FeatureExtractor(
      (model): CLIPModel(
        (text_model): CLIPTextTransformer(
          (embeddings): CLIPTextEmbeddings(
            (token_embedding): Embedding(49408, 512)
            (position_embedding): Embedding(77, 512)
          )
          (encoder): CLIPEncoder(
            (layers): ModuleList(
              (0-11): 12 x CLIPEncoderLayer(
                (self_attn): CLIPSdpaAttention(
                  (k_proj): Linear(in_features=512, out_features=512, bias=True)
                  (v_proj): Linear(in_features=512, out_features=512, bias=True)
                  (q_proj): Linear(in_features=512, out_features=512, bias=True)
                  (out_proj): Linear(in_features=512, out_features=512, bias=True)
                )
                (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
                (mlp): CLIPMLP(
                  (activation_fn): QuickGELUActivation()
   

In [15]:
# # Cell 8 — Optional XAI boundary. 'none' returns exactly zero and preserves FOA baseline.
# def xai_regularizer(adv_image: Tensor, foa_score: Tensor, mode: str = XAI_MODE, weight: float = XAI_WEIGHT) -> Tensor:
#     """Return an optional XAI research term; use ``none``/0 for paper-parity FOA.

#     ``saliency`` penalizes diffuse input gradients. It is deliberately opt-in and changes the method.
#     """
#     if mode == 'none' or weight == 0.0: return foa_score.new_zeros(())
#     if mode == 'saliency':
#         saliency = torch.autograd.grad(foa_score, adv_image, retain_graph=True, create_graph=True)[0].abs()
#         return weight * saliency.flatten(1).std(dim=1).mean()
#     raise ValueError(f'Unknown XAI mode: {mode}')

# print('XAI mode:', XAI_MODE, '| weight:', XAI_WEIGHT)

## Attack algorithm

In [16]:
# Cell 9 — FGSM FOA optimization loop (editable research boundary).
def foa_fgsm_attack(
    image_org: Tensor, 
    image_tgt: Tensor, 
    extractor: nn.Module, 
    loss_fn: nn.Module, 
    source_crop: nn.Module, 
    target_crop: nn.Module, 
    steps: int = STEPS, 
    epsilon: float = PAPER_EPSILON, 
    alpha: float = PAPER_ALPHA) -> Tensor:
    """Generate a targeted FOA adversarial image using the original iterative FGSM update.

    Input tensors are in [0,255]; returned tensor is clamped to [0,1], exactly as the source script.
    With XAI_MODE='none', loss selection, gradient sign update, projection, and target crops match FOA.
    """
    delta = torch.zeros_like(image_org, requires_grad=True)
    for _ in tqdm(range(steps), desc='FOA attack'):
        with torch.no_grad(): 
            loss_fn.set_ground_truth(target_crop(image_tgt))
        adv_image = image_org + delta
        
        global_features, local_features = extractor(adv_image)
        global_score = loss_fn(global_features, local_features)
        
        if isinstance(source_crop, nn.Identity):
            objective = global_score
        else:
            crop_global, crop_local = extractor(source_crop(adv_image))
            objective = loss_fn(crop_global, crop_local)
        # objective = objective + xai_regularizer(adv_image, objective)
        gradient = torch.autograd.grad(objective, delta, create_graph=False)[0]
        delta.data = torch.clamp(delta + alpha * torch.sign(gradient), min=-epsilon, max=epsilon)
    return torch.clamp((image_org + delta) / 255.0, 0.0, 1.0)

SOURCE_CROP = transforms.RandomResizedCrop(PAPER_INPUT_RES, scale=PAPER_CROP_SCALE)
TARGET_CROP = transforms.RandomResizedCrop(PAPER_INPUT_RES, scale=PAPER_CROP_SCALE)
print(SOURCE_CROP)
print(TARGET_CROP)

RandomResizedCrop(size=(224, 224), scale=(0.5, 0.9), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
RandomResizedCrop(size=(224, 224), scale=(0.5, 0.9), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)


In [17]:
# Cell 10 — Parity guard: validate notebook update against original source on a deterministic mock.
def assert_paper_defaults() -> None:
    """Fail early if a paper-parity run has changed an original FOA default."""
    assert (PAPER_EPSILON, PAPER_ALPHA, PAPER_STEPS, PAPER_CLUSTER_NUMBER) == (16.0, 1.0, 300, 10)
    assert PAPER_BACKBONES == ('B16', 'B32', 'Laion') and PAPER_CROP_SCALE == (0.5, 0.9)

def verify_update_parity() -> None:
    """Compare original and notebook FGSM tensor updates with the same mock feature/loss contract."""
    assert_paper_defaults()
    import ast
    source = (REPO_DIR / 'generate_adversarial_samples_foa_attack.py').read_text(encoding='utf-8')
    assert 'delta + cfg.optim.alpha * torch.sign(grad)' in source and 'min=-cfg.optim.epsilon' in source
    class MockExtractor(nn.Module):
        """Minimal differentiable feature extractor for parity testing only."""
        def forward(self, x):
            """Return mean global/local features matching the FOA extractor contract."""
            return {0: x.mean((2, 3)).squeeze(0)}, {0: x.mean((2, 3)).unsqueeze(1)}
    class MockLoss(nn.Module):
        """Target-feature similarity loss for the deterministic parity test only."""
        def set_ground_truth(self, x):
            """Cache mean target features."""
            self.target = x.mean((2, 3)).detach()
        def forward(self, global_f, local_f):
            """Return a differentiable score whose ascent moves input toward target."""
            return -(global_f[0] - self.target.squeeze(0)).square().mean()
    x, y = torch.full((1, 3, 4, 4), 100.0, device=DEVICE), torch.full((1, 3, 4, 4), 130.0, device=DEVICE)
    ours = foa_fgsm_attack(x, y, MockExtractor(), MockLoss(), nn.Identity(), nn.Identity(), steps=3, epsilon=16, alpha=1)
    tree = ast.parse(source); original_node = next(node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == 'fgsm_attack')
    namespace = {'torch': torch, 'nn': nn, 'tqdm': lambda sequence, **_: sequence, 'log_metrics': lambda *_, **__: None, 'MainConfig': object, 'Optional': Optional, 'transforms': transforms}
    exec(compile(ast.Module(body=[original_node], type_ignores=[]), '<pinned-foa>', 'exec'), namespace)
    from types import SimpleNamespace
    cfg = SimpleNamespace(optim=SimpleNamespace(steps=3, epsilon=16, alpha=1), model=SimpleNamespace(use_source_crop=False))
    original = namespace['fgsm_attack'](cfg, MockExtractor(), MockLoss(), nn.Identity(), nn.Identity(), 0, x, y)
    assert torch.allclose(ours, original), 'Notebook FGSM loop differs from pinned FOA source'
    print('Parity guard passed: notebook output equals pinned original FGSM on the deterministic mock.')

verify_update_parity()

FOA attack:   0%|          | 0/3 [00:00<?, ?it/s]

Parity guard passed: notebook output equals pinned original FGSM on the deterministic mock.


# Run Attack

In [ ]:
# Cell 11 — Run attack and write Kaggle-ready PNGs + manifest.
def run_foa_attack() -> Path:
    """Attack paired ImageFolder samples and write source, target, adversarial PNGs and manifest.jsonl."""
    # if XAI_MODE == 'none': 
    #     assert_paper_defaults()
    if OUTPUT_DIR.exists(): 
        shutil.rmtree(OUTPUT_DIR)
    (OUTPUT_DIR / 'images').mkdir(parents=True)
    source_ds = ImageFolderWithPaths(SOURCE_DIR, transform=preprocess_image())
    target_ds = ImageFolderWithPaths(TARGET_DIR, transform=preprocess_image())
    records = []
    
    for index, ((source, _, source_path), (target, _, target_path)) in enumerate(zip(DataLoader(source_ds, batch_size=1, shuffle=False), DataLoader(target_ds, batch_size=1, shuffle=False))):
        if index >= NUM_SAMPLES: 
            break
        source, target = source.to(DEVICE), target.to(DEVICE)
        adversarial = foa_fgsm_attack(source, target, ensemble_extractor, foa_loss, SOURCE_CROP, TARGET_CROP)
        stem = f'{index:05d}'
        paths = {'source': f'images/{stem}_source.png', 'target': f'images/{stem}_target.png', 'adversarial': f'images/{stem}_adversarial.png'}
        save_image(source / 255.0, OUTPUT_DIR / paths['source']); 
        save_image(target / 255.0, OUTPUT_DIR / paths['target']); 
        save_image(adversarial, OUTPUT_DIR / paths['adversarial'])
        
        saved = np.asarray(Image.open(OUTPUT_DIR / paths['adversarial'])).astype(np.int16)
        original = np.asarray(Image.open(OUTPUT_DIR / paths['source'])).astype(np.int16)
        
        linf = int(np.abs(saved - original).max())
        if linf > int(PAPER_EPSILON): 
            raise RuntimeError(f'PNG violates L_inf bound: {linf}')
        records.append({'index': index, 'original_path': source_path[0], 'target_path': target_path[0], 'linf_255': linf, **paths})
    with (OUTPUT_DIR / 'manifest.jsonl').open('w', encoding='utf-8') as handle:
        handle.write('\n'.join(json.dumps(row) for row in records) + ('\n' if records else ''))
    # (OUTPUT_DIR / 'run_config.json').write_text(json.dumps({'num_samples': NUM_SAMPLES, 'steps': STEPS, 'epsilon': PAPER_EPSILON, 'alpha': PAPER_ALPHA, 'xai_mode': XAI_MODE}, indent=2), encoding='utf-8')
    (OUTPUT_DIR / 'run_config.json').write_text(json.dumps({'num_samples': NUM_SAMPLES, 'steps': STEPS, 'epsilon': PAPER_EPSILON, 'alpha': PAPER_ALPHA}, indent=2), encoding='utf-8')
    return OUTPUT_DIR

run_foa_attack()

FOA attack:   0%|          | 0/300 [00:00<?, ?it/s]

running k-means on cuda:0..



[running kmeans]: 0it [00:00, ?it/s]
[running kmeans]: 0it [00:00, ?it/s, center_shift=74.258865, iteration=1, tol=0.000100]
[running kmeans]: 1it [00:00,  8.18it/s, center_shift=74.258865, iteration=1, tol=0.000100]
[running kmeans]: 1it [00:00,  8.18it/s, center_shift=3.001218, iteration=2, tol=0.000100] 
[running kmeans]: 2it [00:00,  8.18it/s, center_shift=0.520647, iteration=3, tol=0.000100]
[running kmeans]: 3it [00:00,  8.18it/s, center_shift=0.047169, iteration=4, tol=0.000100]
[running kmeans]: 4it [00:00,  8.18it/s, center_shift=0.047507, iteration=5, tol=0.000100]
[running kmeans]: 5it [00:00,  8.18it/s, center_shift=0.046906, iteration=6, tol=0.000100]
[running kmeans]: 6it [00:00,  8.18it/s, center_shift=0.016040, iteration=7, tol=0.000100]
[running kmeans]: 8it [00:00, 51.70it/s, center_shift=0.000000, iteration=8, tol=0.000100]


device is :cuda:0
device is :cuda:0
device is :cuda:0
device is :cuda:0
device is :cuda:0
device is :cuda:0
device is :cuda:0
device is :cuda:0
running k-means on cuda:0..



[running kmeans]: 0it [00:00, ?it/s]
[running kmeans]: 0it [00:00, ?it/s, center_shift=256.102966, iteration=1, tol=0.000100]
[running kmeans]: 2it [00:00, 227.92it/s, center_shift=0.000000, iteration=2, tol=0.000100]

...


[running kmeans]: 0it [00:00, ?it/s]
[running kmeans]: 0it [00:00, ?it/s, center_shift=177.143188, iteration=1, tol=0.000100]
[running kmeans]: 1it [00:00, 117.97it/s, center_shift=15.382549, iteration=2, tol=0.000100]
[running kmeans]: 3it [00:00, 206.85it/s, center_shift=0.000000, iteration=3, tol=0.000100] 

device is :cuda:0
device is :cuda:0
device is :cuda:0
running k-means on cuda:0..




[running kmeans]: 0it [00:00, ?it/s]
[running kmeans]: 0it [00:00, ?it/s, center_shift=157.762024, iteration=1, tol=0.000100]
[running kmeans]: 1it [00:00, 138.74it/s, center_shift=6.342990, iteration=2, tol=0.000100]
[running kmeans]: 2it [00:00, 174.93it/s, center_shift=0.547341, iteration=3, tol=0.000100]
[running kmeans]: 3it [00:00, 189.98it/s, center_shift=0.061010, iteration=4, tol=0.000100]
[running kmeans]: 4it [00:00, 200.55it/s, center_shift=0.021743, iteration=5, tol=0.000100]
[running kmeans]: 5it [00:00, 208.36it/s, center_shift=0.015857, iteration=6, tol=0.000100]
[running kmeans]: 7it [00:00, 232.32it/s, center_shift=0.000000, iteration=7, tol=0.000100]


device is :cuda:0
device is :cuda:0
device is :cuda:0
device is :cuda:0
device is :cuda:0
device is :cuda:0
device is :cuda:0
running k-means on cuda:0..



[running kmeans]: 0it [00:00, ?it/s]
[running kmeans]: 0it [00:00, ?it/s, center_shift=92.084656, iteration=1, tol=0.000100]
[running kmeans]: 1it [00:00, 121.69it/s, center_shift=8.747074, iteration=2, tol=0.000100]
[running kmeans]: 2it [00:00, 159.15it/s, center_shift=1.644650, iteration=3, tol=0.000100]
[running kmeans]: 3it [00:00, 178.16it/s, center_shift=0.585969, iteration=4, tol=0.000100]
[running kmeans]: 4it [00:00, 192.84it/s, center_shift=0.913512, iteration=5, tol=0.000100]
[running kmeans]: 5it [00:00, 200.53it/s, center_shift=0.150652, iteration=6, tol=0.000100]
[running kmeans]: 6it [00:00, 205.86it/s, center_shift=0.087078, iteration=7, tol=0.000100]

## Test 1 images

In [ ]:
# def blackbox():
#     """Attack paired ImageFolder samples and write source, target, adversarial PNGs and manifest.jsonl."""
#     # if XAI_MODE == 'none': 
#     #     assert_paper_defaults()
#     if OUTPUT_DIR.exists(): 
#         shutil.rmtree(OUTPUT_DIR)
#     (OUTPUT_DIR / 'images').mkdir(parents=True)
#     source_ds = ImageFolderWithPaths(SOURCE_DIR, transform=preprocess_image())
#     target_ds = ImageFolderWithPaths(TARGET_DIR, transform=preprocess_image())
#     records = []
    
#     for index, ((source, _, source_path), (target, _, target_path)) in enumerate(zip(DataLoader(source_ds, batch_size=1, shuffle=False), DataLoader(target_ds, batch_size=1, shuffle=False))):
#         if index >= NUM_SAMPLES: 
#             break
#         source, target = source.to(DEVICE), target.to(DEVICE)
#         adversarial = foa_fgsm_attack(source, target, ensemble_extractor, foa_loss, SOURCE_CROP, TARGET_CROP)
#         stem = f'{index:05d}'
#         paths = {'source': f'images/{stem}_source.png', 'target': f'images/{stem}_target.png', 'adversarial': f'images/{stem}_adversarial.png'}
#         save_image(source / 255.0, OUTPUT_DIR / paths['source']); 
#         save_image(target / 255.0, OUTPUT_DIR / paths['target']); 
#         save_image(adversarial, OUTPUT_DIR / paths['adversarial'])
        
#         saved = np.asarray(Image.open(OUTPUT_DIR / paths['adversarial'])).astype(np.int16)
#         original = np.asarray(Image.open(OUTPUT_DIR / paths['source'])).astype(np.int16)
        
#         linf = int(np.abs(saved - original).max())
#         if linf > int(PAPER_EPSILON): 
#             raise RuntimeError(f'PNG violates L_inf bound: {linf}')
#         records.append({'index': index, 'original_path': source_path[0], 'target_path': target_path[0], 'linf_255': linf, **paths})
#     with (OUTPUT_DIR / 'manifest.jsonl').open('w', encoding='utf-8') as handle:
#         handle.write('\n'.join(json.dumps(row) for row in records) + ('\n' if records else ''))
#     # (OUTPUT_DIR / 'run_config.json').write_text(json.dumps({'num_samples': NUM_SAMPLES, 'steps': STEPS, 'epsilon': PAPER_EPSILON, 'alpha': PAPER_ALPHA, 'xai_mode': XAI_MODE}, indent=2), encoding='utf-8')
#     (OUTPUT_DIR / 'run_config.json').write_text(json.dumps({'num_samples': NUM_SAMPLES, 'steps': STEPS, 'epsilon': PAPER_EPSILON, 'alpha': PAPER_ALPHA}, indent=2), encoding='utf-8')
#     return OUTPUT_DIR

In [ ]:
# source_image = "/kaggle/input/datasets/zintom/nhut-test-img/nhut.png"
# src_img = Image.open(source_image)
# plt.imshow(src_img)
# plt.axis('off')
# plt.show()

In [ ]:
# test_src_path = '/kaggle/input/datasets/zintom/nhut-test-img/nhut.png'
# test_tar_path = '/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset/images/target_images/1/0.jpg'
# # test_source_ds = ImageFolderWithPaths(test_src_dir, transform=preprocess_image())
# # test_target_ds = ImageFolderWithPaths(test_target_dir, transform=preprocess_image())
# # source_loader = DataLoader(test_source_ds, batch_size=1, shuffle=False)
# # target_loader = DataLoader(test_target_ds, batch_size=1, shuffle=False)

# # source, _, source_path = next(iter(source_loader))
# # target, _, target_path = next(iter(target_loader))
# transform = preprocess_image()
# test_src = Image.open(test_src_path)
# test_src = transform(test_src)

# test_tar = Image.open(test_tar_path)
# test_tar = transform(test_tar)


# test_src.shape, test_tar.shape

# Visualize adversarial images

In [ ]:
adv_img = np.asarray(Image.open('/kaggle/working/foa_attack_output/images/00000_adversarial.png'))

In [ ]:
plt.imshow(adv_img)
plt.axis('off')
plt.show()

In [ ]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# open_ai_key = user_secrets.get_secret("OPENAI_API_KEY")

In [ ]:
# from openai import OpenAI

# client = OpenAI(api_key=open_ai_key)
# response = client.chat.completions.create(
#     model="gpt-4o",
#     messages=[{"role": "user", "content": "Hello, how can I use this API?"}],
# )

# print(response.choices[0].message.content)

In [ ]:
# prompt = f"""Rate the semantic similarity between the following two texts on a scale from 0 to 1.
        
#             **Criteria for similarity measurement:**
#             1. **Main Subject Consistency:** If both descriptions refer to the same key subject or object (e.g., a person, food, an event), they should receive a higher similarity score.
#             2. **Relevant Description**: If the descriptions are related to the same context or topic, they should also contribute to a higher similarity score.
#             3. **Ignore Fine-Grained Details:** Do not penalize differences in **phrasing, sentence structure, or minor variations in detail**. Focus on **whether both descriptions fundamentally describe the same thing.**
#             4. **Partial Matches:** If one description contains extra information but does not contradict the other, they should still have a high similarity score.
#             5. **Similarity Score Range:** 
#                 - **1.0**: Nearly identical in meaning.
#                 - **0.8-0.9**: Same subject, with highly related descriptions.
#                 - **0.7-0.8**: Same subject, core meaning aligned, even if some details differ.
#                 - **0.5-0.7**: Same subject but different perspectives or missing details.
#                 - **0.3-0.5**: Related but not highly similar (same general theme but different descriptions).
#                 - **0.0-0.2**: Completely different subjects or unrelated meanings.
                
#             Text 1: {text1}
#             Text 2: {text2}

#         Output only a single number between 0 and 1. Do not include any explanation or additional text."""

# response = client.chat.completions.create(
#     model=self.model,
#     messages=[{"role": "user", "content": prompt}],
#     max_tokens=100,
#     temperature=0.0,
# )
# score = response.choices[0].message.content.strip()
# print(min(1.0, max(0.0, float(score))))

In [ ]:
# Cell 12 — Preview and ZIP for Kaggle Output/download.
import matplotlib.pyplot as plt

def export_kaggle_zip(output_dir: Path = OUTPUT_DIR) -> Path:
    """Create a ZIP archive that Kaggle exposes under /kaggle/working Output."""
    if not (output_dir / 'manifest.jsonl').exists(): 
        raise FileNotFoundError('Run Cell 11 first.')
    return Path(shutil.make_archive(str(WORK_ROOT / 'foa_attack_output'), 'zip', root_dir=output_dir))

if (OUTPUT_DIR / 'manifest.jsonl').exists():
    record = json.loads((OUTPUT_DIR / 'manifest.jsonl').read_text().splitlines()[0])
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for axis, role in zip(axes, ('source', 'target', 'adversarial')):
        axis.imshow(Image.open(OUTPUT_DIR / record[role])); axis.set_title(role); axis.axis('off')
    plt.show()
    print('ZIP:', export_kaggle_zip())
else:
    print('No output yet. Run Cell 11 with RUN_ATTACK=True.')